In [1]:
from bbs_iss.entities.issuer import IssuerInstance
from bbs_iss.entities.holder import HolderInstance
from bbs_iss.interfaces.credential import VerifiableCredential
import bbs_iss.interfaces.requests_api as api
import bbs_iss.utils.utils as utils

issuer = IssuerInstance()
holder = HolderInstance()

issuer_pub_key = issuer.public_key

print(f"Issuer's BLS public key {len(issuer_pub_key.key)} bytes: {issuer_pub_key.key}")

Issuer's BLS public key 96 bytes: b'\x99\xc7\xf3\x84\x9av\xcb\xe4\xb7"o\x8f\x94\xebJ\x94Q7\x9d\xd1\x96\xe3\x1e?\xf2\xdb1<\xd3;\xe0\x18\xe6Pv\x8f|v\xa3j\x19\x10\x7f\\Lo\x1e\x98\x0e}\x03\xd4\xfcmgr\x8d\xc7\xa6\xf5\x1eKf-W\x8cH\xfb\xab\xe4M\x11\x8b\xde\x16\xb2\x98\xa7&e\x8eM\x87\x89\xfeP"\x96%M\xac\xfdQ\xad\xd2\xaf'


In [2]:
attributes = api.IssuanceAttributes()
attributes.append("name", "Ilya Smut", api.AttributeType.REVEALED)
attributes.append("link_secret", utils.gen_link_secret(), api.AttributeType.HIDDEN)
print("REVEALED")
for a in attributes.attributes:
    print(a.key, a.index, a.message)
print("HIDDEN")
for a in attributes.blinded_attributes:
    print(a.key, a.index, a.message)

REVEALED
name 0 Ilya Smut
HIDDEN
link_secret 1 52b69b08e525b9947c605a3c9264ef89257e0632dcba3a3a3fc588ab8edb2ae3


In [3]:
# Generating issuance request
initial_issuance_request = holder.issuance_request(issuer_pub_key, attributes, "test-credential")
print(initial_issuance_request.request_type)

# Issuer processes issuance request and produces a Freshness response
freshness_response = issuer.process_request(initial_issuance_request)
print(freshness_response.request_type, freshness_response.nonce)

RequestType.ISSUANCE
RequestType.FRESHNESS b'\xfe;FY\xae<\x12\xff\x10\xe4\xcb}I5\x81\xff\xcd\xfc\x81\xe2\x9d\xd8\x8b|H\x1a*\xfc\x12\x08L\xc8'


In [4]:
# Holder processes the freshness response and creates a blind sign request
blnd_sign_req = holder.process_request(freshness_response)
print(blnd_sign_req.request_type)

RequestType.BLIND_SIGN


In [5]:
# Blind sign request contains following values
from bbs_iss.interfaces.requests_api import BlindSignRequest

def print_blind_sign_request(request: BlindSignRequest):
    """Prints the contents of a BlindSignRequest in a human-readable format."""
    print("--- BlindSignRequest Details ---")
    
    print(f"[{type(request).__name__}] Type: {request.request_type.name}")
    print(f"Total Messages: {request.total_messages}")
    
    print("\nRevealed Attributes:")
    if request.revealed_attributes:
        for attr in request.revealed_attributes:
            print(f"  - Index: {attr.index}")
            print(f"    Key: '{attr.key}'")
            print(f"    Message: '{attr.message}'")
    else:
        print("  None")
        
    print("\nMessages with Blinded Indices:")
    if request.messages_with_blinded_indices:
        for attr in request.messages_with_blinded_indices:
            print(f"  - Index: {attr.index}")
            print(f"    Key: '{attr.key}'")
            print(f"    Message: '{attr.message}'")
    else:
        print("  None")
        
    print(f"\nCommitment (hex):")
    print(f"  {request.commitment.hex() if request.commitment else 'None'}")
    
    print(f"\nProof (hex):")
    print(f"  {request.proof.hex() if request.proof else 'None'}")
    print("--------------------------------")

print_blind_sign_request(blnd_sign_req)

--- BlindSignRequest Details ---
[BlindSignRequest] Type: BLIND_SIGN
Total Messages: 3

Revealed Attributes:
  - Index: 0
    Key: 'name'
    Message: 'Ilya Smut'
  - Index: 2
    Key: 'metaHash'
    Message: 'PLACE-HOLDER-METAHASH'

Messages with Blinded Indices:
  - Index: 1
    Key: 'link_secret'
    Message: ''

Commitment (hex):
  8f75f9551c80f62601fac5d631fb25bec1c5367f1aa9ff5e68a8363e6a41c6dd191750f3b8fad9bb8691e7f82a4fa3e5

Proof (hex):
  8f75f9551c80f62601fac5d631fb25bec1c5367f1aa9ff5e68a8363e6a41c6dd191750f3b8fad9bb8691e7f82a4fa3e5347b06e48d1f74e2b7818a5387c4082359b1af0f19fa14ed970d8ae49709d6f7992fdbb4fe60238830a418685116bfa5c802d035290f129dc54a3e75c82cc39aa1e0ad01f534f513a75a48945cd2f8d1000000025889901b5dd56017a066100b22e9c417bfae18e7cebf864e15e9d55e69bbf44661b32ad785f31505d52c16f5413e3247d6744589c0d3cf6e08a24cc5b8693b0e
--------------------------------


In [6]:
# Issuer processes blind sign request and produces ForwardVCResponse
vc_response = issuer.process_request(blnd_sign_req)
print(vc_response.request_type)

# Credential stored within response
print(vc_response.vc.to_json())

RequestType.FORWARD_VC
{
    "@context": [
        "https://www.w3.org/2018/credentials/v1",
        "https://w3id.org/security/bbs/v1"
    ],
    "type": [
        "VerifiableCredential",
        "MockCredential"
    ],
    "issuer": "Mock-Issuer",
    "credentialSubject": {
        "name": "Ilya Smut",
        "link_secret": "",
        "metaHash": "82d4fb1fb733c3bbe6401e0965af4c50bc4da6bfdd0e9266c2c3ec381b76a026"
    },
    "proof": "a624f7cc8e78571f9973c9750e638da30ef87890c4dbafc1298ce481f8f31ba40f4a99abb36fd158d621b15f1c8acdae55463ab6fcdf8b6781c8308a804604bb6234e165e6b5b25c8b163eff7d828f022cfec41d96d4c5594152babab6754b0ec712653b5eb52ac3476b7cf54c400ff6"
}


In [7]:
# Holder processes response, unblinds signature, populates hidden attributes, and verifies the credential
vc_validity_status = holder.process_request(vc_response)
print("Is VC valid?", vc_validity_status)

# Holder now has the following credential
print(holder.credentials["test-credential"].to_json())

Is VC valid? True
{
    "@context": [
        "https://www.w3.org/2018/credentials/v1",
        "https://w3id.org/security/bbs/v1"
    ],
    "type": [
        "VerifiableCredential",
        "MockCredential"
    ],
    "issuer": "Mock-Issuer",
    "credentialSubject": {
        "name": "Ilya Smut",
        "link_secret": "52b69b08e525b9947c605a3c9264ef89257e0632dcba3a3a3fc588ab8edb2ae3",
        "metaHash": "82d4fb1fb733c3bbe6401e0965af4c50bc4da6bfdd0e9266c2c3ec381b76a026"
    },
    "proof": "a624f7cc8e78571f9973c9750e638da30ef87890c4dbafc1298ce481f8f31ba40f4a99abb36fd158d621b15f1c8acdae55463ab6fcdf8b6781c8308a804604bb6234e165e6b5b25c8b163eff7d828f0247cbc5343c128a4b8f8d5bfacdf05e714d940385af7abbb1a1209d25638bc7a5"
}


In [8]:
# Holder can re-verify the VC at any time
holder.verify_vc(vc_name="test-credential", pub_key=issuer_pub_key)

True

In [9]:
# holder can attempt to change any field within the VC
holder.credentials["test-credential"].issuer = "different-issuer"

# but the credential verification will fail due to metaHash value
holder.verify_vc(vc_name="test-credential", pub_key=issuer_pub_key)

False